# Hate Speech Detection — Full Pipeline Demo

**Architecture (no routing):**
```
text → [Layer 2] retrieve neighbors → augment → RAG classifier → label + confidence
     → [Layer 3] LLM explanation → structured moderation output
```

Runs on custom input texts — no ground-truth labels required.  
Edit **Cell 3** to switch model configuration.

In [ ]:
import sys, os, json, re, torch, faiss
import torch.nn.functional as F
import pandas as pd
import numpy as np
from pathlib import Path
from transformers import AutoTokenizer, AutoModel, AutoModelForSequenceClassification

sys.path.insert(0, str(Path("..").resolve()))  # adds src/ so retriever.py is importable
from retriever import retrieve_top_k, retrieve_top_k_above_threshold
from llm_explainer import explain, Layer2Output

## Cell 2 — LLM Backend

In [ ]:
# --- Option A: Groq (free, recommended) — set GROQ_API_KEY in your environment ---
# export GROQ_API_KEY="your_key_here"   (shell)  OR  set it in a .env file
import os
from groq import Groq

groq_key = os.environ.get("GROQ_API_KEY", "")

llm_client = Groq(api_key=groq_key)
LLM_MODEL  = "llama-3.3-70b-versatile"

# --- Option B: Ollama (local, free) ---
# import openai
# llm_client = openai.OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")
# LLM_MODEL = "mistral"

# --- Option C: OpenAI (paid) ---
# import openai
# llm_client = openai.OpenAI(api_key=os.getenv("OPENAI_API_KEY", ""))
# LLM_MODEL = "gpt-4o-mini"

print(f"LLM backend ready: {LLM_MODEL}")

LLM backend ready: llama-3.3-70b-versatile


## Cell 3 — Pipeline Config
`INDEX_SPLIT` controls both which FAISS index is queried and which classifier weights are loaded — they must match.

In [ ]:
MODEL_FAMILY = "roberta"   # "bert" | "roberta"
INDEX_SPLIT  = "full"  # "example" | "knowledge" | "full"
DATASET      = "ISHate"    # "IHC" | "ISHate"
K            = 3           # max neighbors to retrieve
THRESHOLD    = 0.5         # min cosine similarity (sbert is well-separated, not collapsed to ~0.997)

ROOT_DIR        = Path("../..")
INDEX_DIR       = ROOT_DIR / "corpus" / "index"
WEIGHTS_RAC_DIR = ROOT_DIR / "weigths" / "weights_rac_best_hyperparameters"

## Cell 4 — Custom Input Texts

In [ ]:
# ── Add or edit the texts you want to analyse ────────────────────────────────
# No ground-truth labels needed — this is a demo, not a benchmark.

TEXTS = [
    {"id": 1, "text": "Write your first text here."},
    {"id": 2, "text": "Write your second text here."},
    # Add more entries as needed.
]

print(f"Loaded {len(TEXTS)} custom input(s).")
for t in TEXTS:
    print(f"  [{t['id']}] {t['text'][:100]}")

## Cell 5 — Load Pipeline Components

In [ ]:
RETRIEVER_HF_ID = "sentence-transformers/all-mpnet-base-v2"

CLF_HF_IDS = {
    "bert":    "bert-base-uncased",
    "roberta": "roberta-base",
}

def load_pipeline(model_family, index_split, dataset):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Device: {device}")

    # --- Retriever: always sbert, shared across all classifier configs ---
    print(f"Loading retriever: {RETRIEVER_HF_ID} ...")
    ret_tokenizer = AutoTokenizer.from_pretrained(RETRIEVER_HF_ID)
    ret_model     = AutoModel.from_pretrained(RETRIEVER_HF_ID).eval().to(device)

    index_path  = INDEX_DIR / f"vdb_{index_split}.faiss"
    lookup_path = INDEX_DIR / f"lookup_{index_split}.json"
    print(f"Loading index: {index_path} ...")
    index = faiss.read_index(str(index_path))
    with open(lookup_path) as f:
        documents = json.load(f)
    print(f"  Index size: {index.ntotal:,} vectors")

    # --- Classifier: model-family specific, trained on sbert-retrieved data ---
    clf_hf_id = CLF_HF_IDS[model_family]
    clf_path = str(WEIGHTS_RAC_DIR / model_family / "sbert" / index_split / dataset)
    print(f"Loading RAG classifier: {clf_path} ...")
    clf_tokenizer = AutoTokenizer.from_pretrained(clf_path)
    clf_model     = AutoModelForSequenceClassification.from_pretrained(clf_path).eval().to(device)

    print(f"\nReady: {model_family.upper()} | index=sbert/{index_split} | trained_on={dataset}")
    return ret_model, ret_tokenizer, index, documents, clf_model, clf_tokenizer, device


ret_model, ret_tokenizer, index, documents, clf_model, clf_tokenizer, device = load_pipeline(
    MODEL_FAMILY, INDEX_SPLIT, DATASET
)

## Cell 6 — Pipeline Runner and Display

In [ ]:
_LABEL_RE = re.compile(r"^\[(hate|not hate)\]\s*:?\s*", re.IGNORECASE)

def strip_label(text):
    return _LABEL_RE.sub("", text).strip()


def run_pipeline(text, ret_model, ret_tokenizer, index, documents,
                 clf_model, clf_tokenizer, device,
                 llm_client, llm_model, k=K, threshold=THRESHOLD):

    # Layer 2a: retrieve neighbors
    retrieved = retrieve_top_k_above_threshold(
        text, threshold, ret_model, ret_tokenizer, index, documents, chunk_id=None, k=k
    )
    if not retrieved:  # fallback: nothing cleared the threshold
        retrieved = retrieve_top_k(
            text, ret_model, ret_tokenizer, index, documents, chunk_id=None, k=k
        )

    # Layer 2b: augment and classify
    sep = clf_tokenizer.sep_token or "[SEP]"
    augmented = f" {sep} ".join([text] + [t for t, _ in retrieved])
    inputs = clf_tokenizer(
        augmented, return_tensors="pt", truncation=True, padding=True, max_length=256
    )
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        logits = clf_model(**inputs).logits[0]
    probs      = F.softmax(logits, dim=-1)
    pred_idx   = torch.argmax(probs).item()
    label      = "hate" if pred_idx == 1 else "not hate"
    confidence = probs[pred_idx].item()

    # Layer 3: LLM explanation
    l2 = Layer2Output(
        original_text=text, label=label, confidence=confidence,
        hate_category="unknown", retrieved=retrieved
    )
    explanation = explain(l2, llm_client, llm_model)
    return l2, explanation


def display_result(example_id, text, l2, explanation):
    w = 80
    print("=" * w)
    print(f"[{example_id}]  TEXT : {text}")
    print("-" * w)
    print(f"LAYER 2  : {l2.label.upper()}  ({l2.confidence:.1%} confidence)")
    print()
    print(f"RETRIEVED NEIGHBORS ({len(l2.retrieved)}):")
    for i, (txt, score) in enumerate(l2.retrieved, 1):
        print(f"  [{i}] {score:.4f}  {strip_label(txt)[:100]}")
    print()
    print("LAYER 3 EXPLANATION:")
    print(f"  Summary   : {explanation.summary}")
    print(f"  Severity  : {explanation.severity}")
    print(f"  Action    : {explanation.recommended_action}")
    print(f"  Targets   : {', '.join(explanation.target_groups) if explanation.target_groups else chr(8212)}")
    print(f"  Evidence  : {explanation.evidence_used}")
    if explanation.moderator_note:
        print(f"  Note      : {explanation.moderator_note}")
    valid_str = "✓ passed" if explanation.validation_passed else "✗ FAILED (forced human-review)"
    print(f"  Validation: {valid_str}")
    print("=" * w)
    print()

## Cell 7 — Run Full Pipeline on All 50 Examples

In [ ]:
import psutil, time, gc, numpy as np
from concurrent.futures import ThreadPoolExecutor, TimeoutError as FuturesTimeout
from llm_explainer import ExplainerOutput, _extract_label
from retriever import encode

print(f"RAM available: {psutil.virtual_memory().available / 1e9:.1f} GB")
print(f"Config : {MODEL_FAMILY.upper()} | index=sbert/{INDEX_SPLIT} | trained_on={DATASET}")
print(f"Running pipeline on {len(TEXTS)} input(s).\n")

LLM_TIMEOUT      = 30  # seconds per Groq call before giving up

# Unwrap IndexIDMap → extract raw vectors + ID map for pure-numpy cosine search
print("Extracting index vectors for numpy search...", end=" ", flush=True)
_inner  = faiss.downcast_index(index.index)
_xb     = np.empty((index.ntotal, index.d), dtype="float32")
_inner.reconstruct_n(0, index.ntotal, _xb)
_id_map = faiss.vector_to_array(index.id_map).astype("int64")
print(f"done  shape={_xb.shape}")

def retrieve_numpy(text, threshold, k, model, tokenizer):
    """Encode once with mean pooling (sbert), cosine sim via numpy — no FAISS at query time."""
    vec   = encode([text], model, tokenizer, batch_size=1, use_mean_pool=True)
    vec_n = vec / np.maximum(np.linalg.norm(vec, axis=1, keepdims=True), 1e-9)

    sims    = (_xb @ vec_n.T).squeeze()
    top_pos = np.argsort(sims)[::-1][:k]
    top_ids = _id_map[top_pos]
    scores  = sims[top_pos]

    retrieved = [
        (documents[str(int(cid))], float(sc))
        for cid, sc in zip(top_ids, scores)
        if sc >= threshold
    ][:k]
    if not retrieved:
        retrieved = [
            (documents[str(int(cid))], float(sc))
            for cid, sc in zip(top_ids, scores)
        ][:k]

    del vec, vec_n, sims, top_pos, top_ids, scores
    return retrieved

records = []
t_start = time.time()

for i, entry in enumerate(TEXTS):
    text       = str(entry["text"])
    example_id = entry["id"]

    t0 = time.time()
    print(f"[{i+1:02d}/{len(TEXTS)}] id={example_id}  ...", end=" ", flush=True)

    retrieved = retrieve_numpy(text, THRESHOLD, K, ret_model, ret_tokenizer)

    sep       = clf_tokenizer.sep_token or "[SEP]"
    augmented = f" {sep} ".join([text] + [t for t, _ in retrieved])
    inputs    = clf_tokenizer(augmented, return_tensors="pt", truncation=True, padding=True, max_length=256)
    inputs    = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        logits = clf_model(**inputs).logits[0]
    del inputs
    probs      = torch.nn.functional.softmax(logits, dim=-1)
    pred_idx   = torch.argmax(probs).item()
    label      = "hate" if pred_idx == 1 else "not hate"
    confidence = probs[pred_idx].item()
    del logits, probs

    l2 = Layer2Output(
        original_text=text, label=label, confidence=confidence,
        hate_category="unknown", retrieved=retrieved,
    )

    try:
        with ThreadPoolExecutor(max_workers=1) as ex:
            fut         = ex.submit(explain, l2, llm_client, LLM_MODEL)
            explanation = fut.result(timeout=LLM_TIMEOUT)
    except FuturesTimeout:
        print(f"TIMEOUT({LLM_TIMEOUT}s) ", end="", flush=True)
        explanation = ExplainerOutput(
            summary="LLM call timed out.", evidence_used=[], target_groups=[],
            severity="unknown", recommended_action="human-review",
            moderator_note="Groq call exceeded timeout.", validation_passed=False,
        )
    except Exception as e:
        print(f"ERR({e}) ", end="", flush=True)
        explanation = ExplainerOutput(
            summary=f"LLM error: {e}", evidence_used=[], target_groups=[],
            severity="unknown", recommended_action="human-review",
            moderator_note="LLM call raised an exception.", validation_passed=False,
        )

    elapsed = time.time() - t0
    print(f"pred={label.upper():8s}  conf={confidence:.1%}  {elapsed:.1f}s")

    records.append({
        "id"                : example_id,
        "text"              : text,
        "predicted"         : label,
        "confidence"        : round(confidence, 4),
        "n_retrieved"       : len(retrieved),
        "top_sim"           : round(retrieved[0][1], 4) if retrieved else None,
        "retrieved_passages": [
            {"text": strip_label(t), "label": _extract_label(t), "score": round(s, 4)}
            for t, s in retrieved
        ],
        "summary"           : explanation.summary,
        "evidence_used"     : explanation.evidence_used,
        "severity"          : explanation.severity,
        "action"            : explanation.recommended_action,
        "target_groups"     : explanation.target_groups,
        "moderator_note"    : explanation.moderator_note,
        "validation_passed" : explanation.validation_passed,
    })

    gc.collect()

results_df = pd.DataFrame(records)
total = time.time() - t_start
print(f"\nDone. {len(records)} input(s) processed  |  total={total/60:.1f} min")

## Cell 7b — Export LLM Report

Generates `llm_report.html` in the current directory — open it in any browser for a colour-coded report of each input.

In [ ]:
from datetime import datetime
from pathlib import Path

CSS = """
* { box-sizing: border-box; margin: 0; padding: 0; }
body { font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', sans-serif;
       max-width: 960px; margin: 0 auto; padding: 2.5rem 2rem;
       background: #fafafa; color: #222; font-size: 14px; line-height: 1.5; }
h1 { font-size: 1.3rem; font-weight: 600; color: #111; margin-bottom: .2rem; }
.run-meta { color: #888; font-size: .82rem; margin-bottom: 2rem; }

.summary-box { display: flex; gap: 0; border: 1px solid #e5e7eb; border-radius: 8px;
               overflow: hidden; margin-bottom: 2rem; background: white; }
.stat { flex: 1; text-align: center; padding: .9rem .5rem;
        border-right: 1px solid #e5e7eb; }
.stat:last-child { border-right: none; }
.stat .val { font-size: 1.4rem; font-weight: 600; color: #111; }
.stat .lbl { font-size: .72rem; color: #999; text-transform: uppercase;
             letter-spacing: .06em; margin-top: .1rem; }

.card { background: white; border: 1px solid #e5e7eb; border-radius: 8px;
        padding: 1rem 1.25rem; margin-bottom: .75rem;
        border-left: 3px solid #d1d5db; }

.card-header  { margin-bottom: .5rem; }
.card-id      { font-size: .75rem; color: #aaa; font-family: monospace; margin-right: .4rem; }
.card-text    { font-size: .95rem; font-weight: 500; color: #111; }

.badges { display: flex; gap: .35rem; flex-wrap: wrap; margin-bottom: .65rem; align-items: center; }
span.badge { display: inline-block; padding: .1rem .45rem; border-radius: 3px;
             font-size: .73rem; font-weight: 500; white-space: nowrap;
             border: 1px solid transparent; }

.b-hate     { background: #f5f5f5; color: #555; border-color: #d1d5db; }
.b-nothate  { background: #f5f5f5; color: #555; border-color: #d1d5db; }
.b-high     { background: #f5f5f5; color: #374151; border-color: #9ca3af; font-weight: 600; }
.b-medium   { background: #f5f5f5; color: #374151; border-color: #d1d5db; }
.b-low      { background: #f5f5f5; color: #6b7280; border-color: #e5e7eb; }
.b-autoblock{ background: #f0f0f0; color: #111; border-color: #9ca3af; font-weight: 600; }
.b-review   { background: #f5f5f5; color: #374151; border-color: #d1d5db; }
.b-allow    { background: #f5f5f5; color: #6b7280; border-color: #e5e7eb; }
.b-conf     { background: #f5f5f5; color: #374151; border-color: #d1d5db; font-family: monospace; }

.section-lbl { font-size: .7rem; text-transform: uppercase; letter-spacing: .07em;
               color: #bbb; margin: .75rem 0 .3rem; font-weight: 500; }
.passages    { background: #fafafa; border: 1px solid #f0f0f0; border-radius: 5px;
               padding: .6rem .9rem; }
.passage     { font-size: .84rem; margin: .25rem 0; display: flex; gap: .5rem; align-items: baseline; color: #444; }
.p-rank      { font-family: monospace; color: #bbb; min-width: 2rem; }
.p-sim       { font-family: monospace; color: #aaa; min-width: 4rem; }
.p-lhate     { color: #555; font-weight: 600; white-space: nowrap; }
.p-lnothate  { color: #888; white-space: nowrap; }
.p-text      { color: #555; }
.p-cited     { color: #222; font-weight: 600; }

.llm-box  { background: #f7f8fa; border-left: 2px solid #9ca3af; padding: .6rem .9rem;
            border-radius: 0 5px 5px 0; font-size: .88rem; color: #333; margin: .3rem 0; }
.note-box { background: #f9f8f5; border-left: 2px solid #d1c4a0; padding: .5rem .9rem;
            border-radius: 0 5px 5px 0; font-size: .84rem; color: #666; margin: .3rem 0; }
.targets  { font-size: .83rem; color: #888; margin-top: .3rem; }
"""

def _b(text, cls):
    return f'<span class="badge {cls}">{text}</span>'

def _label_badge(lbl):
    return _b(lbl, "b-hate" if lbl == "hate" else "b-nothate")

def _sev_badge(sev):
    return _b(f"severity: {sev}", {"high": "b-high", "medium": "b-medium", "low": "b-low"}.get(sev, "b-review"))

def _action_badge(action):
    return _b(action, {"auto-block": "b-autoblock", "human-review": "b-review", "allow": "b-allow"}.get(action, "b-review"))

def _escape(s):
    return str(s).replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;").replace('"', "&quot;")

def build_card(rec):
    conf_b   = _b(f"{rec['confidence']:.1%}", "b-conf")

    passages_html = ""
    for i, p in enumerate(rec["retrieved_passages"], 1):
        cited     = i in rec["evidence_used"]
        lbl_cls   = "p-lhate" if p["label"] == "hate" else "p-lnothate"
        text_cls  = "p-cited" if cited else "p-text"
        cited_mark = " ·cited" if cited else ""
        passages_html += (
            f'<div class="passage">'
            f'<span class="p-rank">[{i}]</span>'
            f'<span class="p-sim">{p["score"]:.4f}</span>'
            f'<span class="{lbl_cls}">[{p["label"]}]</span>'
            f'<span class="{text_cls}">{_escape(p["text"][:120])}{cited_mark}</span>'
            f'</div>'
        )

    targets_str = ", ".join(_escape(g) for g in rec["target_groups"]) if rec["target_groups"] else "—"
    note_html = (
        f'<div class="note-box">Note: {_escape(rec["moderator_note"])}</div>'
        if rec.get("moderator_note") else ""
    )

    return f"""
<div class="card">
  <div class="card-header">
    <span class="card-id">#{rec['id']}</span>
    <span class="card-text">{_escape(rec['text'])}</span>
  </div>
  <div class="badges">
    Pred: {_label_badge(rec['predicted'])}
    {conf_b}
    {_sev_badge(rec['severity'])}
    {_action_badge(rec['action'])}
  </div>
  <div class="section-lbl">Retrieved evidence · {rec['n_retrieved']} passages · cited = used by LLM</div>
  <div class="passages">{passages_html}</div>
  <div class="section-lbl">LLM Explanation</div>
  <div class="llm-box">{_escape(rec['summary'])}</div>
  <div class="targets">Target groups: {targets_str}</div>
  {note_html}
</div>"""

# ── stats ─────────────────────────────────────────────────────────────────────
n = len(records)
action_counts = {}
for r in records:
    action_counts[r["action"]] = action_counts.get(r["action"], 0) + 1

config_str = f"{MODEL_FAMILY.upper()} · sbert/{INDEX_SPLIT} · trained_on={DATASET} · LLM={LLM_MODEL}"
timestamp  = datetime.now().strftime("%Y-%m-%d %H:%M")

summary_html = f"""
<div class="summary-box">
  <div class="stat"><div class="val">{n}</div><div class="lbl">Total inputs</div></div>
  <div class="stat"><div class="val">{action_counts.get('auto-block',0)}</div><div class="lbl">Auto-block</div></div>
  <div class="stat"><div class="val">{action_counts.get('human-review',0)}</div><div class="lbl">Human-review</div></div>
  <div class="stat"><div class="val">{action_counts.get('allow',0)}</div><div class="lbl">Allow</div></div>
</div>"""

cards_html = "\n".join(build_card(r) for r in records)

html = f"""<!DOCTYPE html>
<html lang="en">
<head>
  <meta charset="UTF-8">
  <meta name="viewport" content="width=device-width, initial-scale=1">
  <title>Hate Speech Detection — LLM Report</title>
  <style>{CSS}</style>
</head>
<body>
  <h1>Hate Speech Detection — LLM Explanation Report</h1>
  <p class="run-meta">{_escape(config_str)} · {timestamp}</p>
  {summary_html}
  {cards_html}
</body>
</html>"""

out = Path("llm_report.html")
out.write_text(html, encoding="utf-8")
print(f"Saved → {out.resolve()}  ({out.stat().st_size // 1024} KB)")

## Cell 8 — Classification Metrics

In [ ]:
print(f"Config : {MODEL_FAMILY.upper()} | index={INDEX_SPLIT} | trained_on={DATASET}")
print(f"Inputs : {len(records)}")
print("=" * 50)
display(results_df[["id", "predicted", "confidence", "n_retrieved", "top_sim", "severity", "action", "validation_passed"]])

## Cell 9 — Results Summary Table

In [ ]:
print(f"\nResults for {len(records)} input(s):")
print(f"{'ID':<6} {'Predicted':<12} {'Confidence':<12} {'Action'}")
print("-" * 50)
for r in records:
    print(f"  {str(r['id']):<4}  {r['predicted'].upper():<12}  {r['confidence']:.1%:<12}  {r['action']}")

action_counts = {}
for r in records:
    action_counts[r["action"]] = action_counts.get(r["action"], 0) + 1
print(f"\nAction breakdown: {action_counts}")
hate_count = sum(1 for r in records if r["predicted"] == "hate")
print(f"Predicted hate: {hate_count}/{len(records)}")

## Cell 10 — Error Analysis

In [ ]:
# ── Inspect individual results ────────────────────────────────────────────────
# Adjust the filter below to explore any subset of results.

flagged = [r for r in records if r["predicted"] == "hate"]
print(f"Flagged as hate: {len(flagged)}/{len(records)}\n")
for r in flagged:
    print(f"[{r['id']}] pred={r['predicted'].upper():8s}  conf={r['confidence']:.1%}  action={r['action']}")
    print(f"       {r['text'][:120]}")
    print()

## Cell 11 — Compare All Configs *(optional — slow, ~45 min on CPU)*

Runs every `(model, index_split, dataset)` combination on the custom input texts and compares predictions.

In [ ]:
# ## Cell 10b — Inspect raw LLM output for one example
# from retriever import encode

# from llm_explainer import build_prompt, call_llm, validate_output, _REQUIRED_FIELDS

# # Pick any entry index to inspect (0 = first input)
# ENTRY_IDX = 0

# entry = TEXTS[ENTRY_IDX]
# text  = str(entry["text"])

# # Retrieve neighbors (same numpy path as Cell 7, with sbert mean pooling)
# vec   = encode([text], ret_model, ret_tokenizer, batch_size=1, use_mean_pool=True)
# vec_n = vec / np.maximum(np.linalg.norm(vec, axis=1, keepdims=True), 1e-9)
# sims    = (_xb @ vec_n.T).squeeze()
# top_pos = np.argsort(sims)[::-1][:K]
# top_ids = _id_map[top_pos]
# scores  = sims[top_pos]
# retrieved = [(documents[str(int(c))], float(s)) for c, s in zip(top_ids, scores)][:K]

# print(f"Text      : {text}")
# print(f"Retrieved : {len(retrieved)} neighbors")
# for i, (t, s) in enumerate(retrieved, 1):
#     print(f"  [{i}] {s:.4f}  {t[:100]}")
# print()

# # Build and print the prompt sent to the LLM
# prompt = build_prompt(text, "hate", 0.99, "unknown", retrieved)
# print("=" * 60)
# print("PROMPT SENT TO LLM:")
# print("=" * 60)
# print(prompt)
# print()

# # Call LLM and show raw response
# print("=" * 60)
# print("RAW LLM RESPONSE:")
# print("=" * 60)
# try:
#     raw = call_llm(prompt, llm_client, LLM_MODEL)
#     import json
#     print(json.dumps(raw, indent=2))
#     print()
#     valid = validate_output(raw, len(retrieved))
#     print(f"validation_passed: {valid}")
#     if not valid:
#         print("Validation failure reasons:")
#         if not _REQUIRED_FIELDS.issubset(raw.keys()):
#             print(f"  Missing fields: {_REQUIRED_FIELDS - raw.keys()}")
#         if not isinstance(raw.get("evidence_used"), list) or not raw.get("evidence_used"):
#             print(f"  evidence_used is empty or not a list: {raw.get('evidence_used')}")
#         else:
#             bad = [i for i in raw["evidence_used"] if not isinstance(i, int) or not (1 <= i <= len(retrieved))]
#             if bad:
#                 print(f"  evidence_used indices out of range [1,{len(retrieved)}]: {bad}")
#         if raw.get("severity") not in {"low", "medium", "high"}:
#             print(f"  Invalid severity: {raw.get('severity')!r}")
#         if raw.get("recommended_action") not in {"auto-block", "human-review", "allow"}:
#             print(f"  Invalid recommended_action: {raw.get('recommended_action')!r}")
# except Exception as e:
#     print(f"LLM call failed: {e}")
#     import traceback; traceback.print_exc()

In [ ]:
# import psutil, time, gc, numpy as np
# from concurrent.futures import ThreadPoolExecutor, TimeoutError as FuturesTimeout
# from llm_explainer import ExplainerOutput
# from retriever import encode
# import traceback

# # Configs: bert/roberta × IHC/ISHate × example/knowledge/full
# ALL_CONFIGS = [
#     (model, split, dataset)
#     for model   in ["bert", "roberta"]
#     for split   in ["example", "knowledge", "full"]
#     for dataset in ["IHC", "ISHate"]
# ]

# def extract_numpy_index(faiss_index):
#     """Unwrap IndexIDMap and return (xb, id_map) as numpy arrays — no index.search() called."""
#     inner  = faiss.downcast_index(faiss_index.index)
#     xb     = np.empty((faiss_index.ntotal, faiss_index.d), dtype="float32")
#     inner.reconstruct_n(0, faiss_index.ntotal, xb)
#     id_map = faiss.vector_to_array(faiss_index.id_map).astype("int64")
#     return xb, id_map

# def classify_numpy(text, xb, id_map, docs, r_model, r_tok, c_model, c_tok, dev, k, threshold):
#     """Full Layer 2 pipeline: sbert mean-pool retrieval + classifier — no FAISS at query time."""
#     vec   = encode([text], r_model, r_tok, batch_size=1, use_mean_pool=True)
#     vec_n = vec / np.maximum(np.linalg.norm(vec, axis=1, keepdims=True), 1e-9)
#     sims    = (xb @ vec_n.T).squeeze()
#     top_pos = np.argsort(sims)[::-1][:k]
#     top_ids = id_map[top_pos]
#     scores  = sims[top_pos]

#     retrieved = [(docs[str(int(c))], float(s)) for c, s in zip(top_ids, scores) if s >= threshold][:k]
#     if not retrieved:
#         retrieved = [(docs[str(int(c))], float(s)) for c, s in zip(top_ids, scores)][:k]
#     del vec, vec_n, sims, top_pos, top_ids, scores

#     sep       = c_tok.sep_token or "[SEP]"
#     augmented = f" {sep} ".join([text] + [t for t, _ in retrieved])
#     inputs    = c_tok(augmented, return_tensors="pt", truncation=True, padding=True, max_length=256)
#     inputs    = {k: v.to(dev) for k, v in inputs.items()}
#     with torch.no_grad():
#         logits = c_model(**inputs).logits[0]
#     del inputs
#     probs  = torch.nn.functional.softmax(logits, dim=-1)
#     label  = "hate" if torch.argmax(probs).item() == 1 else "not hate"
#     del logits, probs
#     return label

# summary_rows = []

# # Load sbert retriever once — shared across all configs
# print(f"Loading shared sbert retriever: {RETRIEVER_HF_ID} ...")
# device_cmp = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# sbert_tok   = AutoTokenizer.from_pretrained(RETRIEVER_HF_ID)
# sbert_model = AutoModel.from_pretrained(RETRIEVER_HF_ID).eval().to(device_cmp)
# print(f"Retriever loaded on {device_cmp}\n")

# cached_indices = {}   # split → (xb, id_map, docs)

# for model_f, split, dset in ALL_CONFIGS:
#     config_name = f"{model_f}/sbert/{split}/{dset}"
#     print(f"Running {config_name} ...", end=" ", flush=True)
#     try:
#         # Load or reuse sbert index for this split
#         if split not in cached_indices:
#             idx_path = str(INDEX_DIR / f"vdb_{split}.faiss")
#             lkp_path = str(INDEX_DIR / f"lookup_{split}.json")
#             idx      = faiss.read_index(idx_path)
#             with open(lkp_path) as fh:
#                 docs = json.load(fh)
#             xb, id_map = extract_numpy_index(idx)
#             cached_indices[split] = (xb, id_map, docs)
#             del idx
#         xb, id_map, docs = cached_indices[split]

#         # Load classifier
#         clf_path = str(WEIGHTS_RAC_DIR / model_f / "sbert" / split / dset)
#         c_tok    = AutoTokenizer.from_pretrained(clf_path)
#         c_model  = AutoModelForSequenceClassification.from_pretrained(clf_path).eval().to(device_cmp)

#         preds = []
#         for entry in TEXTS:
#             pred = classify_numpy(
#                 str(entry["text"]), xb, id_map, docs,
#                 sbert_model, sbert_tok, c_model, c_tok, device_cmp, K, THRESHOLD
#             )
#             preds.append({"id": entry["id"], "predicted": pred})
#         summary_rows.append({"Config": config_name, "Predictions": preds})
#         print(f"done ({len(preds)} inputs)")

#         del c_model, c_tok
#         gc.collect()
#     except Exception as e:
#         print(f"ERROR: {e}")
#         traceback.print_exc()

# if summary_rows:
#     summary_df = pd.DataFrame(summary_rows)
#     display(summary_df)
# else:
#     print("\nNo configs completed — all raised errors (see above).")